In [1]:
using Pkg
Pkg.instantiate()
using QAlgebra
using BenchmarkTools 

Precompiling project...
   2976.0 ms  ✓ QAlgebra
   5206.6 ms  ✓ QAlgebra → QAlgebraPlottingExt
  2 dependencies successfully precompiled in 9 seconds. 299 already precompiled.


In [2]:
subspace_def = SubSpaceDefinitions( h=QubitPM("beta"), 
                                    i=Ensemble(3, 3, QubitPauli("sigma"), as_continuum=true), 
                                    b=Ladder(max_magnitude=4))
op_def = OperatorDefinitions("A(i,t)", "B(U,H,i)")
var_def = ParameterDefinitions( "alpha" => 2.0, 
                                "beta(t)" => t->t^2, 
                                "delta_i" => QUniform(0,1,10), # from 0 to 1 with 10 samples on grid along the gamma dimension
                                "eta_i",
                                "gamma_{i,j}(t, delta_i, delta_j)" => (t, gi, gj)->t*gi+gj,
                                "t")# => 1.0) 
qspace = QSpace(subspace_def, op_def, var_def, max_t_ind=2)
resolve_param!(qspace, "η", QNormal(0,1,2,10))
update_t!(qspace, 1.0)

QSpace: [t,α,β(t),δᵢ,ηᵢ,γᵢˏᵢ(t)]
   - Subspace: h              → PM Qubit (Fermion):  βᵖₚ,βᵐₚ,βᶻₚ
   - Ensemble: i,j,k, ∑ l,m,n → Pauli Qubit (Fermion):  σˣₚ,σʸₚ,σᶻₚ
   - Subspace: b              → Ladder (Boson):  p
   - Abstract Ops:A, B(H,U)


In [3]:
alpha, beta, gamma, delta = base_operators(qspace, ["alpha", "beta", "gamma", "delta"], do_fun=true)
t0, t1 = base_operators(qspace, :t)
ph, mh, zh = base_operators(qspace, "h")
sigma = base_operators(qspace, "i", by_ensemble=true, do_fun=true)  # general constructor for all ensemble indices
xi,yi,zi = base_operators(qspace, "i")
xj, yj, zj = base_operators(qspace, "j")
xk, yk, zk = base_operators(qspace, "k")
xl, yl, zl = base_operators(qspace, "l")
xm, ym, zm = base_operators(qspace, "m")
xn, yn, zn = base_operators(qspace, "n")
b = base_operators(qspace, "b")
I = base_operators(qspace, "I")
A = base_operators(qspace, "A", do_fun=true)
B = base_operators(qspace, "B", do_fun=true)
println("Done")

Done


In [8]:
concrete_indices = ConcreteIndexes(qspace)
pushindex!(concrete_indices, EnsembleIndex(1,3), 2)
pv = qspace.sample_index_param_values
evaluate((delta[:l]).terms[1].coeff_fun, pv, concrete_indices)

6.90677162437177e-310 + 0.0im

In [5]:
qspace.sample_index_param_values

ParameterValues:
  def init group          size     
  ✓   ✓    t              3        
  ✓   ✓    α              1        
  ✓   ✓    β(t)           3        
  ✓   ✓    δᵢ (pdf)       100      
  ✓   ✓    ηᵢ (pdf)       100      
  ✓   ✓    γᵢˏⱼ(t,δᵢ,δⱼ)  3×100×100


In [7]:
p.param_indices

ErrorException: type Array has no field param_indices

In [4]:
qspace.sample_index_param_values[6,1,:,:]

100×100 Matrix{Float64}:
   4.64278e-310  NaN             …    4.64278e-310  NaN
   6.95311e-310    4.64278e-310       6.95311e-310    0.0
   4.64278e-310    4.64278e-310       4.64278e-310    1.0e-323
   6.95198e-310    6.95311e-310       6.95198e-310    0.0
 NaN               1.6976e-313      NaN               4.64278e-310
   0.0             0.0           …    0.0             0.0
   1.0e-323        4.64278e-310       6.95311e-310    4.64278e-310
   0.0             6.95198e-310       0.0             6.95198e-310
   4.64278e-310  NaN                  4.64278e-310  NaN
   0.0             0.0                6.95311e-310    0.0
   ⋮                             ⋱                  
   6.95198e-310    0.0                4.64278e-310    0.0
 NaN               4.64278e-310     NaN               4.64278e-310
 NaN               6.95311e-310       0.0             0.0
   1.0e-323        4.64278e-310       6.95311e-310    4.64278e-310
   0.0             6.95198e-310  …    0.0             6.95198e-3

In [5]:
abstract_params = AbstractIndexParameters(qspace)

ParameterValues:
  def init group          size 
  ✓   ✓    t              3    
  ✓   ✓    α              1    
  ✓   x    β(t)           3    
  ✓   x    δᵢ (pdf)       6    
  ✓   x    ηᵢ (pdf)       6    
  ✓   x    γᵢˏⱼ(t,δᵢ,δⱼ)  3×6×6


In [10]:
update_t!(abstract_params, 0.5; slot=0)

In [11]:
resolve_ensemble_values!(abstract_params, SubSpaceIndex(2,1,3), [2.0, 1.0])

In [16]:
abstract_params[6,0,1,1]

3.0